<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/experimentos_amostra_bin%C3%A1ria/RECONSTRU%C3%87%C3%83O_DO_R%C3%93TULO_MULTICLASSE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Reconstrução do Rótulo Multiclasse a partir das Subclasses

Os experimentos anteriores demonstraram que a classificação binária apresentou desempenho superior à abordagem multiclasse original. Além disso, a análise das probabilidades permitiu identificar uma estrutura interna consistente dentro das classes **Adequada** e **Não adequada**, revelando a existência de diferentes níveis de confiança associados às previsões do modelo.

A divisão das amostras em quatro subgrupos mostrou que os dados não estavam organizados apenas em duas categorias rígidas, mas em uma sequência gradual de qualidade ambiental:

```text
Melhores adequadas
        ↓
Piores adequadas
        ↓
Melhores não adequadas
        ↓
Piores não adequadas
```

A análise das variáveis ambientais e dos modelos treinados para distinguir esses subgrupos indicou que as classes **Piores adequadas** e **Melhores não adequadas** ocupavam uma região de transição entre os extremos de qualidade. Essas amostras apresentavam características intermediárias e estavam mais próximas da fronteira de decisão aprendida pelo modelo binário.

Diante desse resultado, surgiu a hipótese de que a dificuldade observada nos experimentos multiclasse originais estava relacionada à forma como a classe intermediária havia sido construída. Inicialmente, essa classe era definida exclusivamente a partir de intervalos fixos do score ambiental. Embora essa abordagem fosse coerente do ponto de vista das regras utilizadas para construção do rótulo, os resultados mostraram que a separação gerada não refletia adequadamente a estrutura observada nos dados.

Com base nos resultados obtidos pelos modelos binários e pelos experimentos com subclasses, foi proposta uma nova estratégia para reconstruir a classificação em três níveis de qualidade.

Em vez de utilizar diretamente os intervalos do score ambiental, o novo rótulo passou a ser definido a partir da posição de cada amostra em relação à fronteira aprendida pelos modelos.

A nova estrutura foi definida da seguinte forma:

```text
Melhores adequadas        → Adequada

Piores adequadas          → Atenção
Melhores não adequadas    → Atenção

Piores não adequadas      → Não adequada
```

Nessa abordagem, a classe **Atenção** passa a representar explicitamente a região de transição entre condições adequadas e não adequadas. Diferentemente da estratégia anterior, essa categoria não é mais definida apenas por cortes arbitrários no score, mas sim por grupos identificados a partir do comportamento observado pelos modelos de aprendizado de máquina.

O objetivo desse experimento é verificar se a reconstrução da classe intermediária utilizando a fronteira aprendida pelos modelos resulta em uma separação mais consistente entre as categorias. Em outras palavras, busca-se responder à seguinte questão:

> Uma classe intermediária construída a partir da região de transição identificada pelos modelos apresenta melhor desempenho do que uma classe intermediária definida apenas por intervalos fixos do score ambiental?

Para responder a essa pergunta, foi criado um novo rótulo multiclasse composto pelas categorias **Adequada**, **Atenção** e **Não adequada**, que será utilizado nos experimentos supervisionados apresentados a seguir.

Caso essa nova definição produza melhores resultados, isso indicará que a principal limitação observada anteriormente não estava relacionada à capacidade dos algoritmos de aprendizado, mas sim à forma como a classe intermediária havia sido originalmente estruturada.


In [ ]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/Cópia de amostra_binaria_com_subgrupos.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/Cópia de amostra_binaria_com_subgrupos.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (59896, 26)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year,prob_adequada,prob_nao_adequada,subgrupo_qualidade
0,Canada,FISW_32,Lake,2003-12-02,0.043792,2.13333,9.824,0.00200,7.7900,12.00000,...,1,1,2.0,1,5,Adequada,2003,0.971713,0.028287,Melhores adequadas
1,Canada,IEEA_10_32,Lake,2001-06-08,0.015920,0.55000,9.824,0.00400,7.7900,16.80000,...,1,1,2.0,1,5,Adequada,2001,0.971437,0.028563,Melhores adequadas
2,Canada,CHRW-1876,River,2000-01-12,0.064400,10.87500,11.250,0.03590,8.2833,12.76150,...,0,1,1.0,1,4,Não adequada,2000,0.907796,0.092204,Melhores não adequadas
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.071725,1.24444,5.850,0.20425,7.1000,18.32500,...,1,0,3.7,1,4,Não adequada,2004,0.803643,0.196357,Melhores não adequadas
4,Canada,CZPLA_391,River,2003-01-12,0.039740,1.83333,11.050,0.06100,7.7500,8.66667,...,1,0,2.0,1,4,Não adequada,2003,0.907967,0.092033,Melhores não adequadas


In [ ]:
def gerar_novo_rotulo(subgrupo):
    if subgrupo == "Melhores adequadas":
        return "Adequada"
    elif subgrupo in ["Piores adequadas", "Melhores não adequadas"]:
        return "Atenção"
    elif subgrupo == "Piores não adequadas":
        return "Não adequada"

In [ ]:
df["conama_status_revisado"] = df["subgrupo_qualidade"].apply(gerar_novo_rotulo)

In [ ]:
df["conama_status_revisado"].value_counts()

,count
conama_status_revisado,
Atenção,29946
Adequada,20585
Não adequada,9365


In [ ]:
pd.crosstab(
    df["conama_status"],
    df["conama_status_revisado"]
)

conama_status_revisado,Adequada,Atenção,Não adequada
conama_status,,,
Adequada,20585,20584,0
Não adequada,0,9362,9365


In [ ]:
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df["conama_status_revisado"]

In [ ]:
# DIVISÃO TREINO/TESTE
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (47916, 5)
Teste: (11980, 5)


In [ ]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
# sem balanceamento
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1))])

In [ ]:
# MÉTRICAS DE TREINO sem balanceamento
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.9204858502379164
Train Precision:
0.9262303198976285
Train Recall:
0.9204858502379164
Train F1:
0.9210923681889642

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.94      0.98      0.96     16468
     Atenção       0.96      0.87      0.92     23956
Não adequada       0.78      0.93      0.85      7492

    accuracy                           0.92     47916
   macro avg       0.89      0.93      0.91     47916
weighted avg       0.93      0.92      0.92     47916

Train Confusion Matrix:
[[16192   276     0]
 [ 1086 20941  1929]
 [    0   519  6973]]


In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.9042570951585976

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.93      0.97      0.95      4117
     Atenção       0.94      0.86      0.90      5990
Não adequada       0.75      0.89      0.82      1873

    accuracy                           0.90     11980
   macro avg       0.88      0.91      0.89     11980
weighted avg       0.91      0.90      0.91     11980


Confusion Matrix:
[[4010  107    0]
 [ 295 5149  546]
 [   0  199 1674]]


In [ ]:
print(df["prob_adequada"].describe())

count    59896.000000
mean         0.686758
std          0.284501
min          0.029986
25%          0.392714
50%          0.840420
75%          0.933483
max          0.994598
Name: prob_adequada, dtype: float64


In [ ]:
print(df["prob_nao_adequada"].describe())

count    59896.000000
mean         0.313242
std          0.284501
min          0.005402
25%          0.066517
50%          0.159580
75%          0.607286
max          0.970014
Name: prob_nao_adequada, dtype: float64


In [ ]:
print(
    df[df["conama_status"] == "Adequada"]
    ["prob_adequada"]
    .describe()
)

count    41169.000000
mean         0.808587
std          0.211358
min          0.055540
25%          0.767153
50%          0.907923
75%          0.947702
max          0.994598
Name: prob_adequada, dtype: float64


In [ ]:
print(
    df[df["conama_status"] == "Não adequada"]
    ["prob_nao_adequada"]
    .describe()
)

count    18727.000000
mean         0.581069
std          0.237306
min          0.010070
25%          0.429536
50%          0.669155
75%          0.751157
max          0.970014
Name: prob_nao_adequada, dtype: float64


## Resultados da Classificação Multiclasse Reconstruída

Após a reconstrução do rótulo multiclasse a partir das subclasses identificadas pelos modelos binários, foi realizado um novo treinamento utilizando as classes **Adequada**, **Atenção** e **Não adequada**.

Os resultados obtidos foram significativamente superiores aos observados nos experimentos multiclasse anteriores.

No conjunto de treinamento, o modelo alcançou:

* Acurácia: **92,05%**
* Precisão ponderada: **92,62%**
* Recall ponderado: **92,05%**
* F1-Score ponderado: **92,11%**

No conjunto de teste, os resultados permaneceram bastante próximos:

* Acurácia: **90,43%**
* Precisão ponderada: **91,00%**
* Recall ponderado: **90,43%**
* F1-Score ponderado: **91,00%**

A pequena diferença entre treino e teste indica boa capacidade de generalização e sugere que o modelo não apresentou sobreajuste significativo. Isso demonstra que os padrões aprendidos durante o treinamento foram efetivamente transferidos para dados não vistos.

### Análise das Classes

A classe **Adequada** apresentou excelente desempenho. No conjunto de teste, foram obtidos:

* Precisão: **93%**
* Recall: **97%**
* F1-Score: **95%**

Esses resultados indicam que o modelo consegue identificar com elevada confiabilidade as amostras que apresentam melhores condições ambientais.

A classe **Não adequada** também apresentou desempenho expressivo:

* Precisão: **75%**
* Recall: **89%**
* F1-Score: **82%**

Embora a precisão seja inferior à observada para as demais classes, o elevado recall demonstra que o modelo consegue identificar a maior parte das amostras críticas. Do ponto de vista do monitoramento ambiental, essa característica é particularmente importante, pois reduz a probabilidade de que corpos hídricos potencialmente problemáticos sejam classificados como adequados.

A classe **Atenção**, que representa a região intermediária de qualidade, apresentou:

* Precisão: **94%**
* Recall: **86%**
* F1-Score: **90%**

Esse resultado é especialmente relevante porque essa foi justamente a classe que apresentou maiores dificuldades nos experimentos anteriores. A obtenção de métricas elevadas para essa categoria indica que a nova estratégia conseguiu representar de forma mais consistente a região de transição entre condições adequadas e não adequadas.

### Análise da Matriz de Confusão

A matriz de confusão revela um comportamento bastante interessante do modelo.

Observa-se que praticamente não ocorreram confusões diretas entre as classes extremas:

```text
Adequada → Não adequada = 0 ocorrências
Não adequada → Adequada = 0 ocorrências
```

Esse resultado demonstra que o modelo aprendeu de forma consistente a diferença entre os cenários claramente adequados e claramente não adequados.

Os erros observados concentram-se principalmente entre classes vizinhas:

```text
Adequada ↔ Atenção
Atenção ↔ Não adequada
```

Esse comportamento é esperado e desejável, uma vez que a classe **Atenção** foi concebida justamente para representar uma região intermediária de transição. Em termos práticos, isso significa que o modelo não está confundindo situações extremamente distintas, mas apenas amostras que já apresentam características próximas entre si.

Essa característica reforça a coerência da nova estrutura de classificação:

```text
Adequada
     ↓
Atenção
     ↓
Não adequada
```

### Por que os Resultados Melhoraram?

A principal diferença em relação aos experimentos anteriores está na forma como a classe intermediária foi construída.

Na abordagem inicial, a classe intermediária era definida diretamente por intervalos fixos do score ambiental. Embora essa estratégia fosse baseada em critérios previamente estabelecidos, ela não necessariamente refletia a forma como os dados estavam distribuídos no espaço de atributos.

Os experimentos com classificação binária mostraram que existia uma região de transição entre as classes extremas. Essa região foi identificada por meio das probabilidades produzidas pelo modelo e posteriormente representada pelas subclasses:

```text
Piores adequadas
Melhores não adequadas
```

Ao reconstruir a classe **Atenção** utilizando justamente esses grupos de fronteira, o novo rótulo passou a representar uma região que efetivamente existia nos dados.

Em outras palavras, a classe intermediária deixou de ser definida apenas por um corte arbitrário baseado no score e passou a ser construída utilizando informações extraídas diretamente do comportamento dos modelos.

### Interpretação Final

Os resultados indicam que a principal limitação observada anteriormente não estava relacionada à capacidade dos algoritmos de aprendizado de máquina, mas sim à forma como a classe intermediária havia sido originalmente definida.

A reconstrução do rótulo permitiu alinhar a classificação às estruturas naturalmente presentes nos dados, reduzindo a sobreposição entre as categorias e produzindo fronteiras mais consistentes entre as classes.

Dessa forma, a estratégia proposta conseguiu combinar a simplicidade de uma classificação em três níveis com a robustez obtida a partir da análise das probabilidades dos modelos binários.

Além de melhorar o desempenho preditivo, essa abordagem oferece uma interpretação mais coerente da qualidade da água, permitindo distinguir com maior precisão situações claramente adequadas, situações críticas e casos intermediários que merecem atenção e monitoramento.


In [ ]:
df["conama_status"].value_counts()

,count
conama_status,
Adequada,41169
Não adequada,18727


In [ ]:
df["conama_status_revisado"].value_counts()

,count
conama_status_revisado,
Atenção,29946
Adequada,20585
Não adequada,9365


In [ ]:
pd.crosstab(
    df["subgrupo_qualidade"],
    df["conama_status_revisado"]
)

conama_status_revisado,Adequada,Atenção,Não adequada
subgrupo_qualidade,,,
Melhores adequadas,20585,0,0
Melhores não adequadas,0,9362,0
Piores adequadas,0,20584,0
Piores não adequadas,0,0,9365


In [ ]:
df["conama_status"] = df["conama_status_revisado"]

df.to_parquet(
    "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_final.parquet",
    index=False
)